In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/gen-ai-summarization-sprint/samsum-train.csv
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum-test.csv
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum-validation.csv
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/dataset_dict.json
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/validation/state.json
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/validation/dataset_info.json
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/validation/data-00000-of-00001.arrow
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/test/state.json
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/test/dataset_info.json
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/test/data-00000-of-00001.arrow
/kaggle/input/competitions/gen-ai-summarization-sprint/samsum_dataset/train/state.json
/kaggle/input/competitions/

In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/gen-ai-summarization-sprint/samsum-train.csv")
train_data.head(5)

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [3]:
train_data.isnull().sum()

id          0
dialogue    1
summary     0
dtype: int64

In [4]:
train_data.dropna(inplace=True)
train_data.isnull().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [5]:
test_data = pd.read_csv("/kaggle/input/competitions/gen-ai-summarization-sprint/samsum-test.csv")
test_data.head(5)

,id,dialogue,summary
0,13862856,"Hannah: Hey, do you have Betty's number?\nAman...",Hannah needs Betty's number but Amanda doesn't...
1,13729565,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,Eric and Rob are going to watch a stand-up on ...
2,13680171,"Lenny: Babe, can you help me with something?\r...",Lenny can't decide which trousers to buy. Bob ...
3,13729438,"Will: hey babe, what do you want for dinner to...",Emma will be home soon and she will let Will k...
4,13828600,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",Jane is in Warsaw. Ollie and Jane has a party....


In [6]:
test_data.isnull().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [7]:
val_data = pd.read_csv("/kaggle/input/competitions/gen-ai-summarization-sprint/samsum-validation.csv")
val_data.head(5)

,id,dialogue,summary
0,13817023,"A: Hi Tom, are you busy tomorrow’s afternoon?\...",A will go to the animal shelter tomorrow to ge...
1,13716628,Emma: I’ve just fallen in love with this adven...,Emma and Rob love the advent calendar. Lauren ...
2,13829420,Jackie: Madison is pregnant\r\nJackie: but she...,Madison is pregnant but she doesn't want to ta...
3,13819648,Marla: <file_photo>\r\nMarla: look what I foun...,Marla found a pair of boxers under her bed.
4,13728448,Robert: Hey give me the address of this music ...,Robert wants Fred to send him the address of t...


In [8]:
val_data.isnull().sum()

id          0
dialogue    0
summary     0
dtype: int64

In [9]:
print(f"The shape of train_data is {train_data.shape}")
print(f"The shape of test_data is {test_data.shape}")
print(f"The shape of val_data is {val_data.shape}")

The shape of train_data is (14731, 3)
The shape of test_data is (819, 3)
The shape of val_data is (818, 3)


In [10]:
!pip install transformers datasets rouge-score sentencepiece

In [11]:
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset
from rouge_score import rouge_scorer

In [12]:
train_dataset = Dataset.from_pandas(train_data)
val_dataset   = Dataset.from_pandas(val_data)

In [13]:
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [14]:
def preprocess(example):
    inputs = ["summarize: " + text for text in example["dialogue"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        example["summary"],
        max_length=60,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [15]:
train_dataset = train_dataset.map(preprocess, batched=True)
val_dataset   = val_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    #evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,   # keep small for students
    weight_decay=0.01,
    logging_steps=10,
    save_total_limit=1
)

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [18]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,17.975102
20,13.129500
30,9.827406
40,7.239938
50,5.685211
60,4.922728
70,4.280418
80,3.718398
90,3.589436
100,3.739398


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1842, training_loss=2.453473735192182, metrics={'train_runtime': 435.2883, 'train_samples_per_second': 33.842, 'train_steps_per_second': 4.232, 'total_flos': 1993720077484032.0, 'train_loss': 2.453473735192182, 'epoch': 1.0})

In [19]:
def generate_summary(text):
    input_text = "summarize: " + text.strip()
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=60,
        min_length=20,
        num_beams=4,
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [20]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

scores = []
i=0
for _, row in val_data.iterrows():
    pred = generate_summary(row['dialogue'])
    print(i)
    i+=1
    
    s = scorer.score(row['summary'], pred)
    
    score = (
        0.4 * s['rouge1'].fmeasure +
        0.3 * s['rouge2'].fmeasure +
        0.3 * s['rougeL'].fmeasure
    )
    
    scores.append(score)

print("Validation ROUGE Score:", np.mean(scores))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [22]:
test_predictions = []
i=0
for dialogue in test_data['dialogue']:
    print(i)
    pred = generate_summary(dialogue)
    test_predictions.append(pred)
    i+=1

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [23]:
submission = pd.DataFrame({
    "id": test_data["id"],
    "predicted_summary": test_predictions
})

submission.to_csv("submission.csv", index=False)